# ema-second-moment composite — cx28: v_hat per-coordinate scale feeds an in-place param update

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-second-moment`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-second-moment"
DD_ATOM_IDS = ["ema-second-moment", "inplace-param-update"]
DD_SUBTOPICS = ["Optimizer: Adam EMA second moment", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Adam's per-coordinate adaptive learning rate comes from `v_hat = v / (1 - beta2**t)`. The actual parameter step is `p -= lr * g / (sqrt(v_hat) + eps)`. Two atoms have to wire together:

1. **ema-second-moment** — keep `v` as a running EMA of `g*g`, then bias-correct to `v_hat`. `v_hat` quantifies how 'noisy' each coordinate has been historically — large `v_hat` → small effective step on that coordinate.
2. **inplace-param-update** — apply the step to the parameter IN PLACE: `p.data.addcdiv_(g, sqrt(v_hat) + eps, value=-lr)` or `p.data -= lr * g / (sqrt(v_hat) + eps)`. The in-place form is critical when `p` is wrapped as `nn.Parameter` and may have `requires_grad=True` — out-of-place rebinding would break the optimizer's hold on the tensor.

**Anatomy (Adam variant: RMSProp — just the v-side, no momentum).**
```python
v.mul_(beta2).addcmul_(g, g, value=1 - beta2)           # ema-second-moment.
v_hat = v / (1 - beta2 ** t_step)
p.data.addcdiv_(g, v_hat.sqrt().add_(eps), value=-lr)   # inplace-param-update.
```

We test the **RMSProp-style** update (no momentum on the gradient — just adaptive scaling) to isolate the v-to-param wiring. The first-moment is exercised separately in cx25/cx27.

### Composite Exercise — v_hat per-coordinate scale feeds an in-place param update

**Atoms exercised together**: `ema-second-moment`, `inplace-param-update`

Implement `cx28_rmsprop_step(p, v, g, t_step, lr, beta2, eps)`.

Inputs:
- `p` — a `t.Tensor` parameter (may have `requires_grad=True`).
- `v` — running second-moment buffer (same shape as `p`).
- `g` — gradient for this step.
- `t_step` — post-increment step counter (`1` on the first call).
- `lr`, `beta2`, `eps` — hyperparams.

Required behaviour:
1. Update `v` IN PLACE with the second-moment EMA: `v <- beta2*v + (1-beta2)*g**2`.
2. Compute `v_hat = v / (1 - beta2**t_step)` (may be a new scratch tensor).
3. Update `p` IN PLACE: `p -= lr * g / (sqrt(v_hat) + eps)`. The id and data_ptr of `p` MUST be preserved; this is what `inplace-param-update` means.
4. Return `None`.

Both `v` and `p` must be mutated in place. The test confirms:
- `id(p)`, `id(v)` unchanged.
- `p.data_ptr()`, `v.data_ptr()` unchanged.
- After step 1 with unit gradient and `v=0`: `v == 1 - beta2`, `v_hat == 1`, so `p_new == p_old - lr / (1 + eps)`.
- Multi-step trajectory matches the manual formula.

In [ ]:
def cx28_rmsprop_step(p, v, g, t_step, lr, beta2, eps):
    # In-place leaf update requires inference_mode / no_grad.
    with t.inference_mode():
        # Atom A (ema-second-moment): v <- beta2*v + (1-beta2)*g*g.
        v.mul_(beta2).addcmul_(g, g, value=1 - beta2)
        # Bias-correct to v_hat.
        v_hat = v / (1 - beta2 ** t_step)
        # Atom B (inplace-param-update): p -= lr * g / (sqrt(v_hat) + eps).
        p.data.addcdiv_(g, v_hat.sqrt().add_(eps), value=-lr)


<details><summary>Show solution — cx28</summary>

```python
def cx28_rmsprop_step(p, v, g, t_step, lr, beta2, eps):
    # In-place leaf update requires inference_mode / no_grad.
    with t.inference_mode():
        # Atom A (ema-second-moment): v <- beta2*v + (1-beta2)*g*g.
        v.mul_(beta2).addcmul_(g, g, value=1 - beta2)
        # Bias-correct to v_hat.
        v_hat = v / (1 - beta2 ** t_step)
        # Atom B (inplace-param-update): p -= lr * g / (sqrt(v_hat) + eps).
        p.data.addcdiv_(g, v_hat.sqrt().add_(eps), value=-lr)
```

Using `addcdiv_` on `p.data` (not on `p` itself) keeps the autograd machinery quiet — the leaf's `.data` is unwatched. The alternative (`with t.no_grad(): p -= ...`) also works. The key contract `inplace-param-update` enforces is that `p` keeps its identity, because the optimizer's `state` dict and any external references (e.g. a `Module`'s `_parameters` dict) are keyed by tensor identity.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx28'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx28',
        'subtopics': ["Optimizer: Adam EMA second moment", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()